# Search Apotheken Umschau entries

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

root_dir = Path.home() / 'scratch/projects/befunddolmetscher'

In [ ]:
import sys

sys.path.append((root_dir / 'search').as_posix())

### load Apotheken Umschau data

In [ ]:
import json

with (root_dir / 'data/apoum_20251024_conditions.json').open('r') as f:
    apoum = json.load(f)
apoum = {d['instance_id']: d for d in apoum}
len(apoum)

In [ ]:
from sqlitevec import SQLiteVec

apoum_table = {}
for lang in ['de']:
    apoum_table[lang] = SQLiteVec(
        table="apotheken_umschau_doc",
        db_file="apotheken_umschau.db",
        lang=lang,
    )

### load lookup-tables

Download `*.db` from HeiBox and put it in the same directory.

In [ ]:
from helpers import load_dictionary_tables, load_data_tables

icd10_table, mesh_table, do_table, pschyrembel_table = load_dictionary_tables()

In [ ]:
from sqlitevec import SQLiteVec

msd_table = {}
for lang in ['en', 'de']:
    msd_table[lang] = SQLiteVec(
        table=f"msd_doc_{lang}",
        db_file="msd.db",
        lang=lang,
    )

In [ ]:
# placeholder to store the ids
external_ids = {}
for instance_id, r in apoum.items():
    tmp = {k: set([]) for k in ['ICD-10CM', 'MeSH', 'DOID', 'Pschyrembel', 'MSD']}
    for k, v in r['external_ids'].items():
        if isinstance(v, list):
            for u in v:
                if isinstance(u, dict):
                    tmp[k].add(u['value'])
    external_ids[instance_id] = tmp

In [ ]:
external_ids['722949']

In [ ]:
import json
with (root_dir / 'data/apoum_translations.json').open('r') as f:
    translations = json.load(f)

with (root_dir / 'data/apoum_alternative_names.json').open('r') as f:
    alternative_names = json.load(f)

### lookup Apotheken Umschau entries in the dictionaries

In [ ]:
from helpers import lookup_title, _add_to_dict
from tqdm.auto import tqdm

for instance_id, r in tqdm(apoum.items()):
    queries = {'en': [translations[instance_id]], 'de': [r['de']['amateur']['topic']['name']]}
    if instance_id in alternative_names:
        queries['de'].extend(alternative_names[instance_id])

    for keys, dict_table in [
        (['icd10'], icd10_table),
        (['descriptor_ui'], mesh_table),
        (['doid', 'icd10', 'mesh'], do_table),
        (['pschyrembel_id', 'icd10'], pschyrembel_table),
        (['instance_id'], msd_table),
    ]:
        for lang, q in queries.items():
            for query in q:
                if lang in dict_table:
                    hits = lookup_title(table=dict_table[lang], query=query, keys=keys)
                    external_ids = _add_to_dict(hits, external_ids, instance_id)

In [ ]:
external_ids['722949']

### assign found external_ids with distance

In [ ]:
from helpers import search_by_similarity, check_by_similarity
from tqdm.auto import tqdm

for instance_id, r in tqdm(apoum.items()):
    emb = {}
    #for lang in ['de']:
    found = apoum_table['de'].metadata_search(instance_id=instance_id, audience="amateur")
    emb['de'] = found[0]['embedding']
    emb['en'] = found[0]['embedding']

    for name, key, dict_table in [
        ('ICD-10CM', 'icd10', icd10_table),
        ('MeSH', 'descriptor_ui', mesh_table),
        ('DOID', 'doid', do_table),
        ('Pschyrembel', 'pschyrembel_id', pschyrembel_table),
        ('MSD', 'instance_id', msd_table),
    ]:
        if name not in r['external_ids'] or r['external_ids'][name] is None:
            if len(external_ids[instance_id][name]) > 0:
                ids = list(external_ids[instance_id][name])
                distances = check_by_similarity(dict_table, emb, ids, key, limit=25)
            else:
                distances = search_by_similarity(apoum_table, dict_table, emb, instance_id, key, limit=5)
            r['external_ids'][name] = [{'value': k, 'distance': v} for k, v in distances.items()] if len(distances) > 0 else None


In [ ]:
apoum['722949']['external_ids']

In [ ]:
with (root_dir / 'data/apoum_20251029.json').open('w') as f:
    json.dump([r for _, r in apoum.items()], f, indent=4)

### add texts to the database

In [ ]:
import json
output_path = (root_dir / 'results/apoum_embed.jsonl')

vectors = [json.loads(l) for l in output_path.read_text().splitlines()]
vectors = {(v['instance_id'], v['language'], v['edition'], v['paragraph_id']): v['embedding'][0] for v in vectors if len(v['embedding'][0]) == 4096}
len(vectors)

In [ ]:
from tqdm.auto import tqdm

titles = []
texts = []
metadatas = []
embeds = []

for instance_id, r in tqdm(apoum.items()):
    d = r['de']['amateur']
    titles.append(d['topic']['name'])
    texts.append(d['plain_text'])
    metadatas.append({'instance_id': instance_id, 'language': 'de', 'audience': 'amateur'})
    embeds.append(vectors[(instance_id, 'de', 'amateur', d['paragraphs'][0]['paragraph_id'])])

In [ ]:
from sqlitevec import SQLiteVec

apoum_table = {}
for lang in ['de']:
    apoum_table[lang] = SQLiteVec.from_texts(
        titles=titles,
        texts=texts,
        embeds=embeds,
        metadatas=metadatas,
        table="apotheken_umschau_doc",
        db_file="apotheken_umschau.db",
        lang=lang,
    )